In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import random

In [2]:
def load_data(path='worldcities.csv'):
    df = pd.read_csv(path)
    words = df['city_ascii'].dropna().tolist()
    print(f"Loaded {len(words)} city names")

    special_token = '<s>'
    vocab = [special_token] + sorted(set(''.join(words)))
    stoi = {s: i for i, s in enumerate(vocab)}
    itos = {i: s for s, i in stoi.items()}
    return words, vocab, stoi, itos

words, vocab, stoi, itos = load_data()
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
words[:5]

Loaded 48057 city names
Vocab size: 64


['Tokyo', 'Jakarta', 'Delhi', 'Guangzhou', 'Mumbai']

In [3]:
def build_dataset(words, block_size):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in list(w) + ['<s>']:
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X, Y = torch.tensor(X), torch.tensor(Y)
    return X, Y

def split_data(words, block_size, splits=(0.8, 0.9)):
    random.seed(42)
    random.shuffle(words)
    X, Y = build_dataset(words, block_size)
    n1 = int(splits[0] * X.shape[0])
    n2 = int(splits[1] * X.shape[0])
    Xtr, Xdev, Xte = X.tensor_split([n1, n2])
    Ytr, Ydev, Yte = Y.tensor_split([n1, n2])
    print(f"Xtr: {Xtr.shape}, Ytr: {Ytr.shape}")
    print(f"Xdev: {Xdev.shape}, Ydev: {Ydev.shape}")
    print(f"Xte: {Xte.shape}, Yte: {Yte.shape}")
    return (Xtr, Ytr), (Xdev, Ydev), (Xte, Yte)

In [4]:
def init_model(cfg):
    g = torch.Generator().manual_seed(cfg['seed'])
    bs, ed, hs = cfg['block_size'], cfg['emb_dim'], cfg['hidden_size']
    C  = torch.randn((vocab_size, ed), generator=g)
    W1 = torch.randn((ed * bs, hs), generator=g)
    b1 = torch.randn(hs, generator=g)
    W2 = torch.randn((hs, vocab_size), generator=g)
    b2 = torch.randn(vocab_size, generator=g)
    params = [C, W1, b1, W2, b2]
    for p in params:
        p.requires_grad = True
    print(f"Parameters: {sum(p.nelement() for p in params)}")
    return params

def forward(X, C, W1, b1, W2, b2):
    emb = C[X]
    h = torch.tanh(emb.view(-1, W1.shape[0]) @ W1 + b1)
    logits = h @ W2 + b2
    return logits

@torch.no_grad()
def loss_on_dataset(params, dataset, sample_size=None):
    C, W1, b1, W2, b2 = params
    X, Y = dataset
    if sample_size is not None and sample_size < X.shape[0]:
        ix = torch.randint(0, X.shape[0], (sample_size,))
        X, Y = X[ix], Y[ix]
    logits = forward(X, C, W1, b1, W2, b2)
    loss = F.cross_entropy(logits, Y)
    return loss.item()

def train(cfg, params, train_set, dev_set=None):
    C, W1, b1, W2, b2 = params
    Xtr, Ytr = train_set
    history = []
    for i in range(cfg['train_steps']):
        ix = torch.randint(0, Xtr.shape[0], (cfg['batch_size'],))
        logits = forward(Xtr[ix], C, W1, b1, W2, b2)
        loss = F.cross_entropy(logits, Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        lr = cfg['lr_high'] if i < cfg['lr_switch'] else cfg['lr_low']
        for p in params:
            p.data += -lr * p.grad

        step = i + 1
        if step % cfg['eval_interval'] == 0 or step == cfg['train_steps']:
            train_loss = loss_on_dataset(params, train_set, cfg['eval_train_size'])
            dev_loss = loss_on_dataset(params, dev_set) if dev_set is not None else None
            history.append({'step': step, 'train_loss': train_loss, 'dev_loss': dev_loss})
            if dev_loss is None:
                print(f"Step {step:>7d}/{cfg['train_steps']}  train loss: {train_loss:.4f}")
            else:
                print(f"Step {step:>7d}/{cfg['train_steps']}  train loss: {train_loss:.4f}  dev loss: {dev_loss:.4f}")
    if history and history[-1]['dev_loss'] is not None:
        best = min(history, key=lambda row: row['dev_loss'])
        print(f"Best dev loss: {best['dev_loss']:.4f} at step {best['step']}")
    return history

def evaluate(params, datasets):
    losses = {}
    for name, dataset in datasets.items():
        loss = loss_on_dataset(params, dataset)
        losses[name] = loss
        print(f"{name:10s} loss: {loss:.4f}")
    return losses

def sample(cfg, params, count=None):
    C, W1, b1, W2, b2 = params
    g = torch.Generator().manual_seed(cfg['seed'] + 10)
    for _ in range(count or cfg['sample_count']):
        out, context = [], [0] * cfg['block_size']
        while True:
            emb = C[torch.tensor([context])]
            h = torch.tanh(emb.view(1, -1) @ W1 + b1)
            logits = h @ W2 + b2
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            if ix == 0:
                break
            out.append(ix)
        print(''.join(itos[i] for i in out))

In [5]:
# --- CONFIG (tweak and re-run from here) ---
CONFIG = dict(
    block_size=5,           # context length
    emb_dim=20,             # embedding dimensions
    hidden_size=300,        # hidden layer neurons
    train_steps=200_000,    # training iterations
    batch_size=64,
    lr_high=0.1,            # learning rate for first half
    lr_low=0.01,            # learning rate for second half
    lr_switch=100_000,       # step to switch lr
    eval_interval=10_000,    # validate every N training steps
    eval_train_size=10_000,  # train examples used for a stable train-loss estimate
    seed=2147483647,
    sample_count=20,
)
train_set, dev_set, test_set = split_data(words, CONFIG['block_size'])
params = init_model(CONFIG)

Xtr: torch.Size([387684, 5]), Ytr: torch.Size([387684])
Xdev: torch.Size([48461, 5]), Ydev: torch.Size([48461])
Xte: torch.Size([48461, 5]), Yte: torch.Size([48461])
Parameters: 50844


In [6]:
history = train(CONFIG, params, train_set, dev_set)

Step   10000/200000  train loss: 2.9770  dev loss: 3.0381
Step   20000/200000  train loss: 2.7711  dev loss: 2.8025
Step   30000/200000  train loss: 2.7152  dev loss: 2.7445
Step   40000/200000  train loss: 2.7664  dev loss: 2.8013
Step   50000/200000  train loss: 2.6356  dev loss: 2.6764
Step   60000/200000  train loss: 2.7555  dev loss: 2.7598
Step   70000/200000  train loss: 2.6442  dev loss: 2.6710
Step   80000/200000  train loss: 2.7721  dev loss: 2.7992
Step   90000/200000  train loss: 2.6323  dev loss: 2.6862
Step  100000/200000  train loss: 2.6376  dev loss: 2.6744
Step  110000/200000  train loss: 2.4742  dev loss: 2.5256
Step  120000/200000  train loss: 2.5000  dev loss: 2.5253
Step  130000/200000  train loss: 2.4607  dev loss: 2.5234
Step  140000/200000  train loss: 2.4883  dev loss: 2.5212
Step  150000/200000  train loss: 2.4711  dev loss: 2.5222
Step  160000/200000  train loss: 2.4858  dev loss: 2.5226
Step  170000/200000  train loss: 2.5138  dev loss: 2.5205
Step  180000/2

In [7]:
evaluate(params, {"train": train_set, "val": dev_set})

train      loss: 2.4766
val        loss: 2.5167


{'train': 2.476583242416382, 'val': 2.516706943511963}

In [8]:
evaluate(params, {"test": test_set})

test       loss: 2.5243


{'test': 2.524332284927368}

In [9]:
sample(CONFIG, params)

Daudlenn
Er Porto Elalra
Artof
Wastaka
Biuyadple
Vulgakam
Halhera
Clu
Ba Cuneac
Ailace
Prurmizil
Carpura
Van
Grgete
Aluepi
Caroi
Annomadea
Siurtuin
Dro Jeao do Mardhur
Aeffirt
